# Notebook 01: Bronze Ingestion

Ingests raw data into the Unity Catalog Volume (`/Volumes/performance_vs_toxicity/bronze/raw_files`):
1. **FPL Data**: Downloads player rosters (`players.csv`) and matchweek data (`gameweeks.csv`) from Fantasy Premier League archives.
2. **Reddit Comments**: Fetches historical comments from `r/LiverpoolFC` via the Arctic Shift API with pagination, retry/backoff, and automatic checkpoint resumption.

In [ ]:
import os
import sys
import time
import json
from pathlib import Path

# Ensure project root is available on sys.path dynamically
repo_root = str(Path(os.getcwd()).resolve())
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.common.config import load_config
from src.ingestion.fetch_fpl_season import _download_csv
from src.ingestion.fetch_reddit_dump import fetch_subreddit_comments, _last_created_utc

cfg = load_config()
RAW_DIR = "/Volumes/performance_vs_toxicity/bronze/raw_files"

# --- 1. Ingest FPL Data (players.csv & gameweeks.csv per season) ---
for season_id, season_cfg in cfg["seasons"].items():
    fpl_dir = season_cfg["fpl_season_dir"]
    team_name = cfg["team"]["fpl_team_name"]
    base_url = cfg["fpl"]["base_url"]

    players = _download_csv(f"{base_url}/{fpl_dir}/players_raw.csv")
    teams = _download_csv(f"{base_url}/{fpl_dir}/teams.csv")
    team_id = int(teams[teams["name"] == team_name].iloc[0]["id"])
    team_players = players[players["team"] == team_id]

    gws = _download_csv(f"{base_url}/{fpl_dir}/gws/merged_gw.csv")
    team_gws = gws[gws["team"] == team_name]

    out_dir = Path(f"{RAW_DIR}/fpl/{season_id}")
    out_dir.mkdir(parents=True, exist_ok=True)
    team_players.to_csv(out_dir / "players.csv", index=False)
    team_gws.to_csv(out_dir / "gameweeks.csv", index=False)
    print(f"[{season_id}] FPL ingested: {len(team_players)} players, {len(team_gws)} gameweek records")

# --- 2. Ingest Reddit Comments (comments.jsonl per season, resilient & resumable) ---
subreddit = cfg["team"]["reddit_subreddit"]
reddit_cfg = cfg["reddit"]

MAX_OUTER_RETRIES = 15
OUTER_COOLDOWN_SECONDS = 300

for season_id, season_cfg in cfg["seasons"].items():
    out_dir = Path(f"{RAW_DIR}/reddit/{season_id}")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "comments.jsonl"

    for outer_attempt in range(1, MAX_OUTER_RETRIES + 1):
        resume_from = _last_created_utc(out_path)
        existing_content = out_path.read_text(encoding="utf-8") if resume_from is not None else ""
        start_override = (resume_from + 1) if resume_from is not None else None

        new_lines, count, failed = [], 0, False
        try:
            for comment in fetch_subreddit_comments(
                subreddit=subreddit,
                start_date=season_cfg["start_date"],
                end_date=season_cfg["end_date"],
                base_url=reddit_cfg["base_url"],
                page_size=reddit_cfg["page_size"],
                delay_seconds=reddit_cfg["request_delay_seconds"],
                start_override_epoch=start_override,
            ):
                new_lines.append(json.dumps(comment))
                count += 1
                if count % 500 == 0:
                    print(f"[{season_id}] {count} new comments fetched (attempt {outer_attempt})...")
        except RuntimeError as e:
            failed = True
            print(f"[{season_id}] Batch interrupted after {count} new comments: {e}")

        # Always persist fetched data, whether full or interrupted
        with open(out_path, "w", encoding="utf-8") as f:
            if existing_content:
                f.write(existing_content.rstrip("\n") + "\n")
            if new_lines:
                f.write("\n".join(new_lines) + "\n")
        print(f"[{season_id}] Saved {count} new comments to {out_path}")

        if not failed:
            break
        if outer_attempt < MAX_OUTER_RETRIES:
            print(f"[{season_id}] Waiting {OUTER_COOLDOWN_SECONDS}s before retrying...")
            time.sleep(OUTER_COOLDOWN_SECONDS)
    else:
        print(f"[{season_id}] Maximum retries reached ({MAX_OUTER_RETRIES}). Progress saved; run again to continue.")